## Runtime Compatibility Check

Run this notebook from the repo root or from a Kaggle working directory that contains the repo. The notebook is intentionally thin and delegates training, evaluation, and metrics work to the rebuild-track scripts.


In [ ]:
import os
import platform
import shutil
import sys
import zipfile
from pathlib import Path


EXTRACT_ROOT = Path("/kaggle/working/HNDSR-Rebuild")
INPUT_ROOT = Path("/kaggle/input")


def has_repo_layout(path: Path) -> bool:
    return (path / "src").exists() and (path / "configs").exists() and (path / "scripts").exists()


def materialize_repo(source: Path) -> Path:
    working_candidates = [EXTRACT_ROOT, EXTRACT_ROOT / "Mini Project"]
    for candidate in working_candidates:
        if has_repo_layout(candidate):
            print(f"  Reusing writable repo copy at {candidate}")
            return candidate

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"  Copying repo from {source} -> {EXTRACT_ROOT}")
    shutil.copytree(source, EXTRACT_ROOT, dirs_exist_ok=True)

    for candidate in working_candidates:
        if has_repo_layout(candidate):
            print(f"  Writable repo root: {candidate}")
            return candidate
    return source


def extract_repo_archive(archive_path: Path) -> Path | None:
    extracted_candidates = [EXTRACT_ROOT / "Mini Project", EXTRACT_ROOT]
    for candidate in extracted_candidates:
        if has_repo_layout(candidate):
            print(f"  Reusing extracted repo at {candidate}")
            return candidate

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"  Extracting {archive_path} -> {EXTRACT_ROOT}")
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(EXTRACT_ROOT)

    for candidate in extracted_candidates:
        if has_repo_layout(candidate):
            print(f"  Extracted repo root: {candidate}")
            return candidate
    return None


def iter_input_directories(max_depth: int = 3) -> list[Path]:
    directories = [INPUT_ROOT]
    for candidate in sorted(INPUT_ROOT.rglob("*")):
        if candidate.is_dir() and len(candidate.relative_to(INPUT_ROOT).parts) <= max_depth:
            directories.append(candidate)
    return directories


def discover_input_candidates() -> list[Path]:
    candidates: list[Path] = []
    if not INPUT_ROOT.exists():
        return candidates

    dataset_dirs = iter_input_directories(max_depth=3)
    print(f"  /kaggle/input search roots: {[str(path) for path in dataset_dirs[:20]]}")
    for dataset_dir in dataset_dirs:
        if dataset_dir == INPUT_ROOT:
            continue
        try:
            contents = [child.name for child in list(dataset_dir.iterdir())[:10]]
            print(f"  {dataset_dir}: contents={contents}")
        except Exception as exc:
            print(f"  {dataset_dir}: error listing contents={exc}")

        archive = dataset_dir / "Mini Project.zip"
        if archive.exists():
            extracted = extract_repo_archive(archive)
            if extracted is not None:
                candidates.append(extracted)

        candidates.extend([dataset_dir / "Mini Project", dataset_dir])
    return candidates


def find_repo_root() -> Path:
    candidates = [
        *discover_input_candidates(),
        EXTRACT_ROOT / "Mini Project",
        EXTRACT_ROOT,
        Path("/kaggle/working/Mini Project"),
        Path("/kaggle/working"),
        Path.cwd(),
    ]

    print("Searching for repo root...")
    seen: set[str] = set()
    for candidate in candidates:
        marker = str(candidate)
        if marker in seen:
            continue
        seen.add(marker)

        exists = candidate.exists()
        has_repo = exists and has_repo_layout(candidate)
        print(f"  {candidate}: exists={exists}, has_repo={has_repo}")
        if exists:
            try:
                contents = [child.name for child in list(candidate.iterdir())[:10]]
                print(f"    Contents: {contents}")
            except Exception as exc:
                print(f"    Error listing: {exc}")
        if has_repo:
            if candidate.is_relative_to(INPUT_ROOT):
                return materialize_repo(candidate)
            return candidate

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return Path.cwd()


REPO_ROOT = find_repo_root()
PYTHON = sys.executable
print(f"\nRepo root: {REPO_ROOT}")
print(f"Python: {PYTHON}")
print(f"Platform: {platform.platform()}")
print(f"Kaggle runtime type: {os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'local-or-unknown')}")
assert (REPO_ROOT / 'src').exists() and (REPO_ROOT / 'configs').exists(), 'Expected standalone rebuild repo under repo root.'

## Post-Restart GPU Sanity Check

Run the next cell after any runtime restart. If CUDA is not available, continue on CPU only for debugging and document the limitation in the handoff note.


In [ ]:
import torch

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Active GPU: {gpu_name}")
    print(f"CUDA capability: {capability[0]}.{capability[1]}")
    if 'P100' in gpu_name:
        print('Warning: Kaggle assigned Tesla P100. Continue in compatibility mode, but do not treat this as preferred benchmark hardware.')
    else:
        print('Preferred benchmark path: non-legacy CUDA hardware detected.')
else:
    print('Running without CUDA. Keep this run diagnostic-focused if you stay on CPU.')


# vR.2 HNDSR

## Supervised Residual Kaggle Baseline

This is the supervised reset notebook after the failed `vR.1` SR3 baseline. It is a Kaggle execution shell around repo-owned code, not a notebook-only model implementation.


## Experiment Registry

| Field | Value |
| --- | --- |
| Notebook stem | `vR.2_HNDSR` |
| Lineage | scratch (`vR.x`) |
| Dataset lane | Kaggle `4x-satellite-image-super-resolution` |
| Control lane | Bicubic |
| Trainable lane | Supervised residual baseline |
| Tracking gate | W&B online via Kaggle secret |
| Promotion gate | Successful Kaggle run plus review doc |


## Paper Lineage and Hypothesis

- Near-term goal: prove the scratch supervised SR pipeline can run reproducibly on Kaggle using repo scripts.
- Medium-term goal: use this stable Kaggle control lane before returning to the paper-first remote-sensing dataset track.
- Hypothesis for `vR.2`: a direct supervised residual baseline should beat the frozen `vR.1` SR3 result and produce qualitatively coherent outputs before diffusion work resumes.


## Dataset and Config Contract

- Full training config: `configs/phase2_supervised_vr2_kaggle.yaml`
- Smoke training config: `configs/phase2_supervised_vr2_smoke.yaml`
- Bicubic control config: `configs/phase0_bicubic_vr2_kaggle_control.yaml`
- Dataset contract: explicit paired LR/HR loader, fixed `4x` scale, repo-relative artifact paths
- Run-name contract: `vR.2-control`, `vR.2-smoke`, `vR.2-smoke-eval`, `vR.2-train`, `vR.2-eval`


## Table of Contents

1. Runtime diagnostics
2. W&B setup
3. Notebook readiness validation
4. Bicubic control evaluation
5. Smoke train plus smoke eval
6. Full train plus full eval
7. Results dashboard
8. Troubleshooting and handoff


## Weights & Biases Setup

This notebook now requires authenticated W&B tracking. If Kaggle secrets do not provide `WANDB_API_KEY`, the run must stop and be retried only after the secret is configured.


In [ ]:
import os
import subprocess

VERSION = 'vR.2'
NOTEBOOK_STEM = 'vR.2_HNDSR'
CONFIGS = {
    'control': 'configs/phase0_bicubic_vr2_kaggle_control.yaml',
    'smoke': 'configs/phase2_supervised_vr2_smoke.yaml',
    'train': 'configs/phase2_supervised_vr2_kaggle.yaml',
}
RUN_NAMES = {
    'control': 'vR.2-control',
    'smoke': 'vR.2-smoke',
    'smoke_eval': 'vR.2-smoke-eval',
    'train': 'vR.2-train',
    'eval': 'vR.2-eval',
}
ARTIFACT_ROOT = REPO_ROOT / 'artifacts'


def run_command(args):
    print(' '.join(str(arg) for arg in args))
    completed = subprocess.run(args, cwd=REPO_ROOT, text=True)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')


def _get_secret(name, aliases=None):
    aliases = aliases or [name]
    for candidate in aliases:
        value = os.environ.get(candidate)
        if value:
            return value
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(candidate)
        except Exception:
            continue
    return None


wandb_key = _get_secret('WANDB_API_KEY', aliases=['WANDB_API_KEY', 'wandb_api_key'])
if not wandb_key:
    raise RuntimeError('Declining Kaggle run: missing WANDB_API_KEY secret. Configure the Kaggle secret, then rerun vR.2.')
os.environ['WANDB_API_KEY'] = wandb_key
os.environ['HNDSR_WANDB_MODE'] = 'online'
os.environ['HNDSR_REQUIRE_WANDB_AUTH'] = '1'
print('W&B key detected. Authenticated online tracking is now enforced for this run.')


## Training Execution

Run the readiness validator before touching training. Do not skip the control lane or the smoke lane on the first Kaggle pass.


In [ ]:
run_command([
    PYTHON,
    'scripts/validate_notebook_version.py',
    '--version', VERSION,
    '--notebook', 'notebooks/versions/vR.2_HNDSR.ipynb',
    '--doc', 'docs/notebooks/vR.2_HNDSR.md',
    '--review', 'reports/reviews/vR.2_HNDSR.review.md',
    '--config', CONFIGS['train'],
    '--smoke-config', CONFIGS['smoke'],
    '--control-config', CONFIGS['control'],
])


In [ ]:
run_command([
    PYTHON,
    'scripts/evaluate_run.py',
    '--config', CONFIGS['control'],
    '--run-name', RUN_NAMES['control'],
])


In [ ]:
run_command([
    PYTHON,
    'scripts/train_baseline.py',
    '--config', CONFIGS['smoke'],
    '--run-name', RUN_NAMES['smoke'],
])

SMOKE_CHECKPOINT = ARTIFACT_ROOT / RUN_NAMES['smoke'] / 'checkpoints' / 'vR.2_supervised_smoke.pt'
print(f'Smoke checkpoint: {SMOKE_CHECKPOINT}')


In [ ]:
run_command([
    PYTHON,
    'scripts/evaluate_run.py',
    '--config', CONFIGS['smoke'],
    '--run-name', RUN_NAMES['smoke_eval'],
    '--checkpoint', str(SMOKE_CHECKPOINT),
])


In [ ]:
run_command([
    PYTHON,
    'scripts/train_baseline.py',
    '--config', CONFIGS['train'],
    '--run-name', RUN_NAMES['train'],
])

FULL_CHECKPOINT = ARTIFACT_ROOT / RUN_NAMES['train'] / 'checkpoints' / 'vR.2_supervised_best.pt'
print(f'Full checkpoint: {FULL_CHECKPOINT}')


## Evaluation and Export

This stage should export the same fixed smoke-pack comparisons and JSON summaries that the review pass will inspect later.


In [ ]:
run_command([
    PYTHON,
    'scripts/evaluate_run.py',
    '--config', CONFIGS['train'],
    '--run-name', RUN_NAMES['eval'],
    '--checkpoint', str(FULL_CHECKPOINT),
])


## Results Dashboard

The next cell reads local JSON summaries and displays the exported comparison grid if it exists.


In [ ]:
import json

from IPython.display import Image, display


def read_summary(run_name: str, filename: str):
    path = ARTIFACT_ROOT / run_name / 'metrics' / filename
    if not path.exists():
        print(f'Missing summary: {path}')
        return None
    payload = json.loads(path.read_text(encoding='utf-8'))
    print(f'===== {run_name} =====')
    print(json.dumps(payload, indent=2))
    return payload


read_summary(RUN_NAMES['control'], 'eval_summary.json')
read_summary(RUN_NAMES['smoke'], 'train_summary.json')
read_summary(RUN_NAMES['smoke_eval'], 'eval_summary.json')
read_summary(RUN_NAMES['train'], 'train_summary.json')
read_summary(RUN_NAMES['eval'], 'eval_summary.json')

grid_path = ARTIFACT_ROOT / RUN_NAMES['eval'] / 'samples' / 'vR.2_supervised_grid.png'
if grid_path.exists():
    display(Image(filename=str(grid_path)))
else:
    print(f'Grid not found yet: {grid_path}')


## Troubleshooting and Known Failure Modes

- If `validate_notebook_version.py` fails, stop and fix the notebook or paired docs before training.
- If Kaggle cannot see the repo paths, inspect `REPO_ROOT` in the first setup cell and adjust the working directory before rerunning.
- If W&B is not authenticated, decline the run immediately, add the Kaggle `WANDB_API_KEY` secret, and rerun the same version.
- If the smoke checkpoint does not appear, inspect the printed command output before moving to the full run.


## Changelog

- `vR.2`: supervised scratch reset after the failed `vR.1` SR3 baseline.
- Notebook-only logic is intentionally limited to diagnostics, subprocess orchestration, and results display.


## Next Step Gate

Do not fork `vR.2` until the executed `vR.2` notebook, its paired metrics, and its review doc all agree on one of these outcomes:

- freeze `vR.2` and promote to the next version only if the supervised baseline is credible
- patch `vR.2` in place for runtime-only or training-contract fixes
- preserve a failure snapshot and document why the supervised reset is not promotable yet
